# Session 1 — Exercises

Three of them, tagged by difficulty. **Do them before Saturday** — the ⭐⭐ one is the scaffold
session 2 builds on, so skipping it costs you tomorrow.

| | Exercise | What it's really teaching |
|---|---|---|
| ⭐ | Token detective | measure, don't assume |
| ⭐⭐ | Ticket loader | the dict-update pattern that returns in S10 |
| ⭐⭐⭐ | Multimodal triage | combining two capabilities into one useful thing — and surveying the field |

Then three **interview questions** at the bottom. Write your answers down before the review
session — reading someone else's answer feels like learning and isn't.

**Rules of engagement:** getting stuck is the point, but getting stuck for an hour is not. Twenty
minutes on one exercise, then look at the solution and re-type it yourself.

## Setup

Run this once. If it fails, run `uv run python check_setup.py` from `labs/` — it tells you exactly
what to fix.

In [ ]:
import os
from pathlib import Path

import yaml
from dotenv import load_dotenv
from google import genai

load_dotenv(Path.cwd().parent / ".env")
client = genai.Client()
MODEL = "gemini-3.1-flash-lite"

DATA = Path.cwd().parent / "data"
tickets = yaml.safe_load((DATA / "tickets.yml").read_text(encoding="utf-8"))

print(f"key found: {bool(os.getenv('GEMINI_API_KEY'))} · {len(tickets)} tickets · model {MODEL}")

---
## ⭐ Exercise 1 — Token detective

Cost, speed and memory are all counted in **tokens**, so knowing what's expensive is a real
engineering skill rather than trivia.

### Part A — predict, in writing, before you run anything

Three texts of roughly equal length: English prose, Bangla prose, Python code. **Which costs the most
tokens per character?**

Write your prediction in the `prediction` variable below, and one sentence on *why*. Committing to a
guess before measuring is the whole exercise — it's how you find out what you believe.

> **My prediction:** …
>
> **Because:** …

### Part B — now measure

Fill in the `count_tokens` call. The API is
`client.models.count_tokens(model=MODEL, contents=text).total_tokens`.

In [ ]:
samples = {
    "English": "Where is my order? I ordered headphones last week and the tracking has not moved.",
    "Bangla":  "আমার অর্ডার কোথায়? আমি গত সপ্তাহে হেডফোন অর্ডার করেছি এবং ট্র্যাকিং আপডেট হয়নি।",
    "Python":  "def triage(text: str) -> dict:\n    return {'category': 'refund', 'urgency': 3}",
}

prediction = "???"          # <- "English", "Bangla" or "Python"
print(f"my prediction for most tokens per character: {prediction}\n")

print(f"{'':10} {'tokens':>7} {'chars':>7} {'chars/token':>13}")
for name, text in samples.items():
    n = None                # TODO: count the tokens in `text`
    if n is None:
        print(f"{name:10} {'TODO':>7}")
        continue
    print(f"{name:10} {n:>7} {len(text):>7} {len(text) / n:>13.2f}")

### Part C — explain the result

Were you right? Whether or not you were, answer these:

1. Which text was most expensive per character, and what is it about that text that fragments into
   more tokens?
2. **Many people believe Bangla costs several times more than English.** Your measurement either
   supports that or contradicts it. If it contradicts it, where do you think the belief came from?
3. ShopWise's inbox is roughly 60% English, 30% Bangla, 10% pasted order JSON. Which of those three
   would you look at first if the monthly bill doubled?

### Part D — the number that actually bites

`count_tokens` measures what you *send*. Go back to §7.7 of the session notebook, where a six-word
follow-up question cost several hundred input tokens.

Answer in one sentence: **for a support chat that runs 20 turns, which grows faster — the number of
messages, or the total tokens billed?** Say why.

> …

---
## ⭐⭐ Exercise 2 — Ticket loader

Open `warmup.py`. It ends at a `# Your turn` comment — this is that turn.

### Part A — count the tickets in each category

Build a dict mapping category → count, then print a small report. Use
`counts.get(key, 0) + 1`, which reads as *"whatever's there, or zero if nothing, plus one"*.

**Remember this shape.** In session 10 it is literally how a node in a graph updates state.

In [ ]:
counts: dict[str, int] = {}

for t in tickets:
    pass    # TODO: one line — add 1 to counts[t["category"]]

if not counts:
    print("counts is still empty — fill in the loop above.")
else:
    for category, n in sorted(counts.items(), key=lambda kv: -kv[1]):
        print(f"{category:18} {'█' * n} {n}")

### Part B — who writes in the most?

Same pattern, different key: count tickets per customer (`from`). Which customer is most expensive to
support? Print them in descending order.

In [ ]:
by_customer: dict[str, int] = {}

# TODO: same pattern as Part A, keyed on t["from"]

print(by_customer or "TODO")

### Part C — put it back in `warmup.py`

Move your working code into `warmup.py`, under the `# Your turn` comment, as a function:

```python
def count_by(tickets: list[dict], field: str) -> dict[str, int]:
    ...
```

Then `count_by(tickets, "category")` and `count_by(tickets, "from")` both work from one function.
Run it with `uv run python warmup.py` and check the output still makes sense.

*Why this matters:* you just wrote a function that takes the **name of a field** as an argument
instead of hard-coding it. That's the difference between a script and a tool — and session 5 is
entirely about writing tools.

---
## ⭐⭐⭐ Exercise 3 — Multimodal triage

In the session you saw two capabilities separately: **§7.2** turned a ticket into validated JSON,
and **§7.3** read a customer's photograph. Neither is very useful alone. Put them together and you
have something ShopWise would actually deploy: *a warranty claim arrives with a photo attached, and
your code decides what happens to it.*

This is the shape of almost every real LLM feature — not one clever call, but two ordinary ones
combined so that the output is something a program can act on.

### Part A — build it

Write **one** call that receives the ticket text *and* the photo, and returns a validated object.
Start from the `PhotoClaim` model below. You have everything you need in §7.2 and §7.3 of the
session notebook; look, don't remember.

Two hints, because these are the parts people get stuck on and neither is interesting:

- `input=` takes a **list** of blocks when you send more than text — one `{"type": "text", …}` and
  one `{"type": "image", …}`.
- The image goes in base64-encoded:
  `base64.b64encode(PHOTO.read_bytes()).decode("utf-8")`.

In [ ]:
import base64
from typing import Literal

from pydantic import BaseModel, Field

PHOTO = DATA / "images" / "ticket-photo-headphones.jpg"
claim_text = (
    "Subject: left earcup padding torn\n"
    "I've had these headphones about four months. The padding on the left cup has "
    "split open and the foam is coming out. Photo attached. Is this covered?"
)


class PhotoClaim(BaseModel):
    """What ShopWise needs decided before a human ever opens the ticket."""

    product: str = Field(description="What the photo shows, in three words or fewer")
    visible_damage: str = Field(description="One line: what is actually wrong in the image")
    likely_cause: Literal["manufacturing_defect", "normal_wear", "accidental_damage", "cannot_tell"]
    confidence: Literal["low", "medium", "high"]
    route_to: Literal["auto_approve", "human_review", "request_more_photos", "decline"]


# TODO: one client.interactions.create(...) call that sends BOTH claim_text and the
#       photo, with response_format set from PhotoClaim.model_json_schema().
#       Then parse it with PhotoClaim.model_validate_json(...).

result = None
print(result or "TODO — build the call above")

### Part B — take the photo away

Run the *same* prompt with the image block removed, so the model has only the customer's words.
Compare the two `PhotoClaim` objects.

Which fields changed, and which stayed the same? **Did `confidence` go down when you removed the
evidence?** (It very often doesn't — and noticing that is the point of Part C.)

In [ ]:
# TODO: same call, text only — no image block. Parse into PhotoClaim and compare.

result_no_photo = None
print(result_no_photo or "TODO")

### Part C — the part that matters

`route_to: "auto_approve"` means money moves without a human. Answer honestly:

1. Your model just assigned a `likely_cause` and a `confidence`. **What could it not possibly know
   from a photograph?** Name at least two things.
2. If Part B produced high confidence *without* the photo, what does that tell you about how much the
   `confidence` field is worth?
3. Would you let this code auto-approve a refund today? If not — what would have to be added first?
   (You are describing sessions 5, 9 and 11. Guess anyway; guessing first makes them land harder.)

> …

### Part D — survey the field

Twenty minutes with three browser tabs. Written answers, no code.

**1. `arena.ai/leaderboard`** — open the overall board, then a category board (coding, or vision).
   - Name one model that ranks noticeably differently between the two boards.
   - Why might a model be strong at one and mediocre at the other?

**2. `huggingface.co`** — search for something in your own language or domain (try `bangla`).
   - Find one model and one dataset. Write down each name and its **licence**.
   - Open a model card. Name one limitation its authors admit to.

**3. `openrouter.ai/models`** — the price list for hundreds of models in one place.
   - Find the cheapest model you'd consider for ticket triage, and the price per million tokens.
   - Compare it to a frontier model on the same page. **How many times more expensive is it?**
   - Using your §8.2 measurement (roughly 100 tokens per triaged ticket), estimate the monthly cost of
     each at 200 tickets/day. Is the expensive one worth it *for this task*?

> …

That last question is the one you will be asked in a real job, and "we used the best model" is the
wrong answer. The right one names a task, a measurement and a budget.

---
## 💼 Interview questions

These are asked in real junior-AI-engineer interviews. Write your answer **before** the review
session — and write it as you'd actually say it out loud, not as bullet points.

### 1. A stakeholder asks: "why don't we just use the best model?"

They've seen a leaderboard. Talk them through how you'd actually choose a model for ShopWise's ticket
triage, in about a minute. You may not say "it depends" without immediately saying *on what*.

> …

### 2. Compute the cost of one call.

Your prompt is **2,000 tokens** and the answer is **500 tokens**. From this price sheet:

| | per 1M tokens |
|---|---|
| input | $0.10 |
| output | $0.40 |

- What does this single call cost?
- What does it cost for 5,000 calls a day, over a 30-day month?
- Your manager wants to halve that bill. **Which of the two numbers do you attack, and why?**

> …

### 3. Your client is a hospital. They ask whether their patient data will be safe.

Explain, without jargon, what the closed-weights vs open-weights choice means for them, and what you
would recommend. Mention at least one thing you would need to know before answering properly.

> …

---
## Checking yourself

Worked solutions with full explanations are published **after the exercise review** at the start
of the next session. Two rules for when they land:

1. **Write something first**, even if it's wrong. A wrong answer you wrote is worth more than a right
   one you read.
2. After reading a solution, **close it and re-type the code from memory.** If you can't, you haven't
   learned it yet — and that's useful information, not a failure.

Bring anything that stayed confusing to the start of Saturday's session. The first ten minutes are
for exactly that.